In [ ]:
import sys
sys.path.append("../src")

from actuarial_loss_prediction.pipelines.feature_engineering.nodes import setup_feature_engineering_pipeline
import yaml

with open("../conf/base/parameters_feature_engineering.yml", "r") as f:
    fe_params = yaml.safe_load(f)


pipeline = setup_feature_engineering_pipeline(fe_params["feature_engineering_parameters"])


/Users/bettina/Documents/Data repository/Actuarial loss prediction/actuarial-loss-prediction/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import pandas as pd

df_train = pd.read_csv("../data/01_raw/train.csv", parse_dates=["DateTimeOfAccident", "DateReported"])

X_train = df_train.drop(fe_params["feature_engineering_parameters"]["target_name"], axis=1)
y_train = df_train[fe_params["feature_engineering_parameters"]["target_name"]]

X_train = pipeline.fit_transform(X_train, y_train)


[SentenceEmbedding] Skipping fit (loading embeddings from ../data/02_intermediate/embeddings/desc_embeddings.db)
[SentenceEmbedding] Loading embeddings from ../data/02_intermediate/embeddings/desc_embeddings.db ...
[SentenceEmbedding] Loaded embeddings shape: (54000, 1025)
[PCA] Fitting PCA (n_components=100)...
[PCA] PCA fitted in 7.68s
[PCA] Transforming embeddings with PCA...
[PCA] PCA done in 0.16s
[PCA] PCA output shape: (54000, 100)


In [ ]:
df_train


,ClaimNumber,DateTimeOfAccident,DateReported,Age,Gender,MaritalStatus,DependentChildren,DependentsOther,WeeklyWages,PartTimeFullTime,HoursWorkedPerWeek,DaysWorkedPerWeek,ClaimDescription,InitialIncurredCalimsCost,UltimateIncurredClaimCost
0,WC8285054,2002-04-09 07:00:00+00:00,2002-07-05 00:00:00+00:00,48,M,M,0,0,500.00,F,38.0,5,LIFTING TYRE INJURY TO RIGHT ARM AND WRIST INJURY,1500,4748.203388
1,WC6982224,1999-01-07 11:00:00+00:00,1999-01-20 00:00:00+00:00,43,F,M,0,0,509.34,F,37.5,5,STEPPED AROUND CRATES AND TRUCK TRAY FRACTURE ...,5500,6326.285819
2,WC5481426,1996-03-25 00:00:00+00:00,1996-04-14 00:00:00+00:00,30,M,U,0,0,709.10,F,38.0,5,CUT ON SHARP EDGE CUT LEFT THUMB,1700,2293.949087
3,WC9775968,2005-06-22 13:00:00+00:00,2005-07-22 00:00:00+00:00,41,M,S,0,0,555.46,F,38.0,5,DIGGING LOWER BACK LOWER BACK STRAIN,15000,17786.487170
4,WC2634037,1990-08-29 08:00:00+00:00,1990-09-27 00:00:00+00:00,36,M,M,0,0,377.10,F,38.0,5,REACHING ABOVE SHOULDER LEVEL ACUTE MUSCLE STR...,2800,4014.002925
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53995,WC9370727,2004-08-21 18:00:00+00:00,2004-09-08 00:00:00+00:00,32,F,S,0,0,500.00,F,38.0,5,STRUCK KNIFE LACERATED LEFT MIDDLE FINGER LEFT...,1000,480.493308
53996,WC8396269,2002-04-28 09:00:00+00:00,2002-09-03 00:00:00+00:00,20,F,S,0,0,500.00,F,40.0,5,LEFT HAND LACERATION LEFT SIDE BACK AND LEFT LEG,1000,755.735319
53997,WC3609528,1992-02-28 09:00:00+00:00,1992-03-18 00:00:00+00:00,19,M,S,0,0,283.00,F,40.0,5,METAL SLIPPED ACROSS METAL CUT FINGER,210,418.178461
53998,WC5038565,1995-01-10 07:00:00+00:00,1995-01-31 00:00:00+00:00,24,M,S,0,0,200.00,F,38.0,5,BURN WHILST USING SPANNER LACERATION RIGHT MID...,7500,2695.225700


In [ ]:
from kedro.io import DataCatalog
from kedro_datasets.pandas import CSVDataset

catalog = DataCatalog({
    "df" : CSVDataset(
  filepath= "../data/01_raw/train.csv",
  load_args={
      "parse_dates": ["DateTimeOfAccident", "DateReported"]
  }
)
})


df_kedro = catalog.load("df")


In [ ]:
pd.testing.assert_frame_equal(df_train, df_kedro)
